# Panel sanity checks
Coverage over time, universe size per date, feature distributions and missingness for `data/processed/panel.parquet` (build it with `python -m src.build_panel`). Logic lives in `src/`; this notebook only inspects the output.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter

from src.utils import load_config

cfg = load_config()
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
panel = pd.read_parquet(ROOT / cfg["output"]["dir"] / f"{cfg['output']['name']}.parquet")
dates = panel.index.get_level_values("date")
features = [c for c in panel.columns if not c.startswith(("z_", "fwd_"))]
z_features = [f"z_{c}" for c in features]
labels = [c for c in panel.columns if c.startswith("fwd_ret_") and not c.endswith("_xs")]
print(f"{len(panel):,} rows, {dates.nunique()} dates ({dates.min().date()} to {dates.max().date()})")
panel.head()

3,509,381 rows, 6995 dates (1999-01-04 to 2026-10-23)


ret_1d    ret_5d   ret_21d   ret_63d  ret_126d  ret_252d  \
date       ticker                                                               
1999-01-04 AAMRQ   0.010546  0.011593 -0.142092  0.117595 -0.313292 -0.059746   
           AAPL    0.006536  0.047619  0.144981  0.175573  0.356828  1.545455   
           ABI1    0.007051  0.019431  0.064308  0.480105  0.558887  0.416667   
           ABS    -0.017664 -0.055658  0.047081  0.119352  0.224158  0.343704   
           ABT    -0.005093  0.018155 -0.002638  0.163144  0.148217  0.467142   

                   mom_12_1   vol_21d   vol_63d  beta_252d  ...  z_trend_200d  \
date       ticker                                           ...                 
1999-01-04 AAMRQ   0.095985  0.027424  0.034097   1.404718  ...     -0.829761   
           AAPL    1.223140  0.037332  0.040950   1.218595  ...      1.064286   
           ABI1    0.331068  0.023753  0.021561   0.869347  ...      1.688003   
           ABS     0.283286  0.018683  0.016887   0.489410  ...      0.775640   
           ABT     0.471023  0.021187  0.020332   0.864223  ...      0.641257   

                   fwd_ret_1d  fwd_ret_1d_xs  fwd_terminal_1d  fwd_ret_5d  \
date       ticker                                                           
1999-01-04 AAMRQ     0.059513       0.038185              0.0    0.018528   
           AAPL     -0.037037      -0.058365              0.0    0.064815   
           ABI1      0.002515      -0.018813              0.0   -0.041493   
           ABS       0.007845      -0.013483              0.0   -0.019603   
           ABT      -0.007584      -0.028912              0.0   -0.046101   

                   fwd_ret_5d_xs  fwd_terminal_5d  fwd_ret_21d  \
date       ticker                                                
1999-01-04 AAMRQ        0.016277              0.0    -0.051718   
           AAPL         0.062564              0.0    -0.126543   
           ABI1        -0.043744              0.0     0.126344   
           ABS         -0.021854              0.0    -0.052263   
           ABT         -0.048352              0.0    -0.062207   

                   fwd_ret_21d_xs  fwd_terminal_21d  
date       ticker                                    
1999-01-04 AAMRQ        -0.036414               0.0  
           AAPL         -0.111239               0.0  
           ABI1          0.141648               0.0  
           ABS          -0.036958               0.0  
           ABT          -0.046903               0.0  

[5 rows x 43 columns]

## Universe size per date
S&P 500 members (point in time) that traded that day. Saved to `reports/figures/universe_size.png`.

In [2]:
per_date = panel.groupby(level="date").size()
fig, ax = plt.subplots(figsize=(10, 3))
per_date.plot(ax=ax)
ax.set_ylabel("tickers per date"); ax.set_ylim(per_date.min() - 10, per_date.max() + 10)
fig.savefig(FIGURES / "universe_size.png", dpi=120, bbox_inches="tight")
print(per_date.describe().round(1))

count    6995.0
mean      501.7
std         2.1
min       490.0
25%       500.0
50%       500.0
75%       503.0
max       506.0
dtype: float64


## Coverage over time
Share of rows with a non-missing value, by year. Saved to `reports/figures/coverage_by_year.png`.

In [3]:
coverage = panel[features + labels].notna().groupby(dates.year).mean()
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(coverage.T.to_numpy(), aspect="auto", vmin=0.8, vmax=1.0, cmap="viridis")
ax.set_xticks(range(len(coverage.index)), coverage.index, rotation=90)
ax.set_yticks(range(coverage.shape[1]), coverage.columns)
_ = fig.colorbar(im, label="share non-missing")
_ = ax.set_title("Coverage by year")
fig.savefig(FIGURES / "coverage_by_year.png", dpi=120, bbox_inches="tight")

## Missingness by feature and label

In [4]:
(panel[features + labels].isna().mean() * 100).round(2).sort_values(ascending=False).to_frame("missing %")

,missing %
ret_252d,0.58
mom_12_1,0.58
beta_252d,0.46
dist_high_252d,0.45
trend_200d,0.35
fwd_ret_21d,0.32
ret_126d,0.27
ret_63d,0.13
vol_63d,0.10
fwd_ret_5d,0.09


## Raw feature distributions (1st to 99th percentile)

In [5]:
cols = [c for c in features if not c.startswith(("dollar_volume",))]
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for ax, col in zip(axes.ravel(), cols):
    s = panel[col].dropna()
    s = s[(s >= s.quantile(0.01)) & (s <= s.quantile(0.99))]
    ax.hist(s, bins=60); ax.set_title(col, fontsize=9)

    # Force X-axis ticks to show exactly 2 decimal places
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
for ax in axes.ravel()[len(cols):]:
    ax.axis("off")
fig.tight_layout()

## Standardised features
Per-date mean should be about 0 and std about 1.

In [6]:
by_date = panel[z_features].groupby(level="date")
summary = pd.DataFrame({"mean |avg|": by_date.mean().abs().mean(), "std avg": by_date.std().mean()})
summary.round(4)

,mean |avg|,std avg
z_ret_1d,0.0,1.0
z_ret_5d,0.0,1.0
z_ret_21d,0.0,1.0
z_ret_63d,0.0,1.0
z_ret_126d,0.0,1.0
z_ret_252d,0.0,1.0
z_mom_12_1,0.0,1.0
z_vol_21d,0.0,1.0
z_vol_63d,0.0,1.0
z_beta_252d,0.0,1.0


## Feature correlations (standardised, pooled)
Shows the overlap between related features. Saved to `reports/figures/feature_correlations.png`.

In [ ]:
corr = panel[z_features].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)), [c[2:] for c in corr.columns], rotation=90)
ax.set_yticks(range(len(corr)), [c[2:] for c in corr.index])
_ = fig.colorbar(im)
fig.savefig(FIGURES / "feature_correlations.png", dpi=120, bbox_inches="tight")

## Labels

In [8]:
panel[labels + [f"{c}_xs" for c in labels]].describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
fwd_ret_1d,3508375.0,0.0005,0.0242,-0.9425,-0.0095,0.0004,0.0103,3.4000
fwd_ret_5d,3506136.0,0.0023,0.0523,-0.9882,-0.0209,0.0026,0.0255,5.6000
fwd_ret_21d,3498093.0,0.0095,0.1031,-0.9989,-0.0404,0.0104,0.0592,4.1408
fwd_ret_1d_xs,3508375.0,0.0000,0.0205,-0.9143,-0.0081,-0.0002,0.0079,3.3501
fwd_ret_5d_xs,3506136.0,0.0000,0.0447,-0.9683,-0.0194,-0.0004,0.0190,5.5972
fwd_ret_21d_xs,3498093.0,0.0000,0.0884,-1.0813,-0.0435,-0.0010,0.0419,4.0844
